# 07 · Grelha N × épocas × τ no TF-IDF

Spike E4-S1 do [`docs/backlog.md`](../docs/backlog.md). Os números que saem daqui vão para a secção 16 do [`docs/qualidade_dados.md`](../docs/qualidade_dados.md).

**Porquê uma grelha:** o número ótimo de épocas **depende do N** — o N muda quantas etiquetas existem e quantos exemplos tem cada uma. O τ também depende do N, porque corrige a frequência das etiquetas. Escolher o N com as épocas e o τ fixos podia escolher um N que não seria o melhor com os seus próprios valores (a **interação** entre hiperparâmetros). Por isso testa-se o produto **N × épocas × τ** de uma vez, em vez de um eixo de cada vez. O `alpha` fica fixo em 0,0000001 (E4-S4/S5).

**Como:** as opções e a forma de decidir estão em [`experiencias/e4_s1.yaml`](../experiencias/e4_s1.yaml), escritas antes de correr; este notebook lê esse ficheiro, para não poder decidir com uma lista diferente da que se treinou. São **18 treinos** (3 N × 6 épocas) no pod; o τ **não retreina** o modelo — só muda a ordem das sugestões ao sugerir — por isso cada treino grava um ficheiro por τ e as 3 × 6 × 3 = **54 combinações** saem desses 18 treinos.

**O modelo em vigor é o do E4-S5:** N = 10, 24 épocas e τ = 0,25. É exatamente a célula `N=10, 24 épocas` desta grelha, por isso este notebook também confirma que ela **reproduz** os números do E4-S5 (78,3% no código completo e 84,1% na classe), antes de tirar qualquer conclusão.

**Os treinos** correram num pod da Runpod com 32 processadores (`scripts/correr_experiencia.py ... grelha --paralelo 4 --n-jobs 8`), com o treino completo. Os ficheiros estão em `metrics/e4_s1/`. O teste não se abre aqui.

In [1]:
import json
import re
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
sys.path.insert(0, str(Path.cwd().parent / "scripts"))
import baselines as bl
import comparison as cmp
import correr_experiencia as ce
import evaluation as ev
import tfidf

RAIZ = Path.cwd().parent
DATA = RAIZ / "data" / "processed"
METRICS = RAIZ / "metrics"
EXP = ce.ler(RAIZ / "experiencias" / "e4_s1.yaml")
N_MIN, EPOCAS, TAUS, ALPHA = EXP["n_min"], EXP["epocas"], EXP["tau"], EXP["alpha"]
EM_VIGOR = (10, 24, tfidf.TAU)  # o modelo do E4-S5: N = 10, 24 épocas, τ = 0,25

treino = pd.read_parquet(DATA / "treino.parquet", columns=["codigo"])
val = pd.read_parquet(DATA / "validacao.parquet", columns=["idcontrato", "codigos", "visto_no_treino"])
print(f"treino: {len(treino):,} exemplos · validação: {len(val):,} contratos")
print(f"alpha: {ALPHA:g} · N: {N_MIN} · épocas: {EPOCAS} · τ: {TAUS}")
print(f"combinações: {len(N_MIN) * len(EPOCAS) * len(TAUS)} (de {len(N_MIN) * len(EPOCAS)} treinos)")
print(f"em vigor (E4-S5): N = {EM_VIGOR[0]}, {EM_VIGOR[1]} épocas, τ = {EM_VIGOR[2]:g}")

treino: 478,384 exemplos · validação: 23,945 contratos
alpha: 1e-07 · N: [5, 10, 20] · épocas: [None, 3, 6, 12, 18, 24] · τ: [0.0, 0.25, 0.5]
combinações: 54 (de 18 treinos)
em vigor (E4-S5): N = 10, 24 épocas, τ = 0.25


## Os treinos

Uma linha por treino, lida do `treino.json` de cada um. Os tempos são de treinos que correram ao mesmo tempo no mesmo pod (4 de cada vez, com 8 processadores cada), por isso servem para comparar entre si, não com outros spikes. Repara que o N = 5 é o mais pesado: tem 3 848 etiquetas (contra 3 000 a N = 10 e 2 193 a N = 20).

In [2]:
def nome_pasta(n, ep):
    return ce.pasta_grelha(EXP, n, ep)


info = {(n, ep): json.loads((RAIZ / nome_pasta(n, ep) / "treino.json").read_text(encoding="utf-8"))
        for n in N_MIN for ep in EPOCAS}
if any(i["n_min"] != n or (i["epocas"] or None) != ep for (n, ep), i in info.items()):
    raise ValueError("algum treino.json não corresponde ao (N, épocas) da pasta")
if any(i["validacao"]["contratos"] != len(val) for i in info.values()):
    raise ValueError("algum treino sugeriu para outra validação")
pd.DataFrame({f"N={n} ep={ep if ep is not None else 'defeito'}": {
    "N": i["n_min"], "épocas pedidas": i["epocas"] if i["epocas"] else "regra por defeito",
    "épocas feitas (máx.)": max(i["n_iter"]), "etiquetas": i["etiquetas"],
    "treinar (s)": i["tempo_vetorizar_treinar_s"], "pico de memória (MiB)": i["pico_rss_mb"]}
    for (n, ep), i in info.items()}).T

,N,épocas pedidas,épocas feitas (máx.),etiquetas,treinar (s),pico de memória (MiB)
N=5 ep=defeito,5,regra por defeito,12,3848,638.15,12616
N=5 ep=3,5.0,3.0,3.0,3848.0,280.52,12640.0
N=5 ep=6,5.0,6.0,6.0,3848.0,589.93,12635.0
N=5 ep=12,5.0,12.0,12.0,3848.0,1313.41,12663.0
N=5 ep=18,5.0,18.0,18.0,3848.0,1724.36,12642.0
N=5 ep=24,5.0,24.0,24.0,3848.0,2474.06,12641.0
N=10 ep=defeito,10,regra por defeito,12,3000,519.98,9990
N=10 ep=3,10.0,3.0,3.0,3000.0,235.03,9992.0
N=10 ep=6,10.0,6.0,6.0,3000.0,437.15,10007.0
N=10 ep=12,10.0,12.0,12.0,3000.0,903.07,10010.0


## As sugestões

Um ficheiro por combinação (N, épocas, τ). As verificações são as dos notebooks 04, 05 e 06: uma linha por contrato da validação, alinhada pelo `idcontrato` (nunca pela posição), com 5 códigos distintos de 8 dígitos. Usam `raise` e não `assert`, porque o Python ignora os `assert` quando corre com `-O`.

In [3]:
CODIGO = re.compile(r"\d{8}")


def nome_tau(tau):
    return "sugestoes_validacao.parquet" if tau == 0 else f"sugestoes_validacao_tau{tau:g}.parquet"


def ler(path):
    sug = pd.read_parquet(path, columns=["idcontrato", "sugestoes"])
    sug["idcontrato"] = sug["idcontrato"].astype(str)
    if sug["idcontrato"].duplicated().any() or set(sug["idcontrato"]) != set(val["idcontrato"]):
        raise ValueError(f"{path}: os contratos não são os da validação, um por linha")
    s = sug.set_index("idcontrato")["sugestoes"].reindex(val["idcontrato"]).map(list)
    s.index = val.index
    if not s.map(lambda l: len(set(l)) == ev.TOP_K and all(CODIGO.fullmatch(c) for c in l)).all():
        raise ValueError(f"{path}: há contratos sem {ev.TOP_K} códigos distintos de 8 dígitos")
    return s


por = {(n, ep, t): ler(RAIZ / nome_pasta(n, ep) / nome_tau(t))
       for n in N_MIN for ep in EPOCAS for t in TAUS}
print(f"{len(por)} combinações lidas (de {len(N_MIN) * len(EPOCAS)} treinos) · ok")

54 combinações lidas (de 18 treinos) · ok


**Repetível?** O treino com N = 10 e 24 épocas é o mesmo do E4-S5 (mesmos dados, mesmo `alpha`, mesma semente, mesmo código do modelo). Se as sugestões não forem iguais às do E4-S5 (76,9% no código completo e 82,9% na classe com τ = 0,25, antes das 24 épocas; 78,3% e 84,1% com elas), nenhuma comparação com o E4-S5 vale. Mudou só o número de processadores, que não muda o modelo.

In [4]:
for t in (0.0, tfidf.TAU):
    e4s5 = ler(METRICS / "e4_s5" / "epocas_24" / nome_tau(t))
    iguais = (e4s5 == por[(10, 24, t)]).mean()
    print(f"τ {t:g}: sugestões iguais às do E4-S5 (N=10, 24 épocas): {iguais:.2%}")
    if iguais < 1:
        raise ValueError("o treino com N = 10 e 24 épocas não repetiu o E4-S5")

τ 0: sugestões iguais às do E4-S5 (N=10, 24 épocas): 100.00%

τ 0.25: sugestões iguais às do E4-S5 (N=10, 24 épocas): 100.00%


## As 54 combinações lado a lado

A régua é a do E4-T2, com o mesmo N. Além das métricas da regra (top-5 no código completo e na classe), mostram-se o top-5 no texto novo, o top-1 e o F1 macro no código completo. Ordenado pelo top-5 no código completo.

In [5]:
def linha(s):
    rep = ev.report(s, val["codigos"], val["visto_no_treino"])
    return {
        "top-5 código": rep.loc[("todos", "codigo"), "top5"],
        "top-5 classe": rep.loc[("todos", "classe"), "top5"],
        "top-5 código (novo)": rep.loc[("texto novo", "codigo"), "top5"],
        "top-5 classe (novo)": rep.loc[("texto novo", "classe"), "top5"],
        "top-1 código": rep.loc[("todos", "codigo"), "top1"],
        "F1 macro código": rep.loc[("todos", "codigo"), "f1_macro"],
    }


top = bl.most_common_labels(treino["codigo"], n_min=10)
regua = bl.suggest_most_common(top, val.index)

tab = pd.DataFrame({c: linha(s) for c, s in por.items()}).T
tab.index = pd.MultiIndex.from_tuples(tab.index, names=["N", "epocas", "tau"])
tab = tab.sort_values("top-5 código", ascending=False)

vis = tab.copy()
vis.index = [f"N={n} ep={int(ep) if pd.notna(ep) else 'defeito'} τ={t:g}" for n, ep, t in vis.index]
vis = pd.concat([vis, pd.DataFrame({"régua": linha(regua)}).T])
vis.style.format({c: "{:.1%}" for c in vis.columns if c != "F1 macro código"} | {"F1 macro código": "{:.4f}"})

,top-5 código,top-5 classe,top-5 código (novo),top-5 classe (novo),top-1 código,F1 macro código
N=10 ep=24 τ=0.25,78.3%,84.1%,75.1%,82.2%,57.7%,0.2568
N=10 ep=24 τ=0,78.2%,83.3%,74.9%,81.2%,58.0%,0.2469
N=5 ep=24 τ=0.25,78.1%,83.3%,74.7%,81.2%,57.6%,0.2647
N=5 ep=18 τ=0.25,77.9%,83.3%,74.5%,81.1%,57.9%,0.2651
N=5 ep=24 τ=0,77.8%,82.5%,74.4%,80.2%,58.1%,0.2514
N=5 ep=12 τ=0.25,77.7%,83.2%,74.3%,81.0%,57.5%,0.2657
N=5 ep=defeito τ=0.25,77.7%,82.8%,74.3%,80.6%,57.2%,0.2618
N=5 ep=18 τ=0,77.7%,82.4%,74.3%,80.1%,58.3%,0.2546
N=5 ep=24 τ=0.5,77.7%,83.5%,74.3%,81.4%,56.7%,0.2710
N=5 ep=12 τ=0,77.7%,82.6%,74.2%,80.2%,57.8%,0.2516


## A decisão

O vencedor é a combinação com o melhor top-5 no **código completo**, desde que não perca o top-5 na **classe** (métrica de guarda, por bootstrap) e desde que bata a opção em vigor (o modelo do E4-S5) pela regra de comparação do E4-T1. Em caso de empate, fica o modelo em vigor. Com 54 combinações, alguma ganharia só por sorte: o número do vencedor fica **otimista** (viés de seleção) e confirma-se no teste, na fase 5.

In [6]:
def duelo(a, b):
    res = {}
    for medida in ("codigo", "classe"):
        ra = cmp.weighted_mean(ev.hits(a, val["codigos"], medida, ev.TOP_K))
        rb = cmp.weighted_mean(ev.hits(b, val["codigos"], medida, ev.TOP_K))
        res[medida] = cmp.paired_bootstrap(ra, rb, len(val))
    return res, cmp.choose(res["codigo"], res["classe"], default="a")


melhor = tab["top-5 código"].idxmax()
print(f"melhor na validação (top-5 código): N={melhor[0]} ep={int(melhor[1]) if pd.notna(melhor[1]) else 'defeito'} "
      f"τ={melhor[2]:g} ({tab.loc[melhor, 'top-5 código']:.2%})")
print(f"em vigor (E4-S5): N={EM_VIGOR[0]} ep={EM_VIGOR[1]} τ={EM_VIGOR[2]:g} "
      f"({tab.loc[EM_VIGOR, 'top-5 código']:.2%})\n")

res, escolha = duelo(por[EM_VIGOR], por[melhor])
for medida, r in res.items():
    print(f"  {medida:7s} em vigor: {r['a']:.2%}  melhor: {r['b']:.2%}  "
          f"diferença {r['b'] - r['a']:+.2%}  melhor ganha em {r['b_melhor']:.0%} das amostras")
escolhido = melhor if escolha == "b" else EM_VIGOR
print(f"\n-> fica N={escolhido[0]} ep={int(escolhido[1]) if pd.notna(escolhido[1]) else 'defeito'} τ={escolhido[2]:g}")

print("\nOs concorrentes mais próximos (distintos) contra o modelo em vigor (E4-S5):")
vistos = 0
for c in tab.index:
    if c == EM_VIGOR:
        continue
    r = duelo(por[EM_VIGOR], por[c])[0]["codigo"]
    ep = "defeito" if pd.isna(c[1]) else int(c[1])
    print(f"  N={c[0]} ep={ep} τ={c[2]:g}: código {r['b']:.2%} vs {r['a']:.2%} "
          f"(diferença {r['b'] - r['a']:+.2%}, conta={r['conta']})")
    vistos += 1
    if vistos == 5:
        break

melhor na validação (top-5 código): N=10 ep=24 τ=0.25 (78.35%)
em vigor (E4-S5): N=10 ep=24 τ=0.25 (78.35%)



  codigo  em vigor: 78.35%  melhor: 78.35%  diferença +0.00%  melhor ganha em 0% das amostras
  classe  em vigor: 84.11%  melhor: 84.11%  diferença +0.00%  melhor ganha em 0% das amostras

-> fica N=10 ep=24 τ=0.25

Os concorrentes mais próximos (distintos) contra o modelo em vigor (E4-S5):


  N=10 ep=24 τ=0: código 78.18% vs 78.35% (diferença -0.17%, conta=True)


  N=5 ep=24 τ=0.25: código 78.05% vs 78.35% (diferença -0.30%, conta=True)


  N=5 ep=18 τ=0.25: código 77.94% vs 78.35% (diferença -0.41%, conta=True)


  N=5 ep=24 τ=0: código 77.79% vs 78.35% (diferença -0.56%, conta=True)


  N=5 ep=12 τ=0.25: código 77.75% vs 78.35% (diferença -0.60%, conta=True)


## O escolhido contra o E4-S5 e contra a régua

A avaliação completa (as quatro medidas, todos os contratos e texto novo) do escolhido, ao lado do modelo do E4-S5. E o critério do E4-H1 repetido para o escolhido: continua a bater a régua no top-5 do código completo e da classe, em todos os contratos e no texto novo?

In [7]:
lado = pd.concat({"escolhido": ev.report(por[escolhido], val["codigos"], val["visto_no_treino"]),
                  "E4-S5": ev.report(por[EM_VIGOR], val["codigos"], val["visto_no_treino"])}, axis=1)
lado.style.format("{:.1%}", subset=[c for c in lado.columns if c[1] in ("top1", "top5")]) \
    .format("{:.4f}", subset=[c for c in lado.columns if c[1] == "f1_macro"]) \
    .format("{:,.0f}", subset=[c for c in lado.columns if c[1] == "contratos"])

In [8]:
novo = ~val["visto_no_treino"]
linhas = {}
for grupo, mask in (("todos", slice(None)), ("texto novo", novo)):
    comp = cmp.compare(por[escolhido][mask], regua[mask], val["codigos"][mask])
    for medida in ("codigo", "classe"):
        r = comp.loc[(medida, "top5")]
        linhas[(grupo, medida)] = {"escolhido": r["a"], "régua": r["b"], "conta": r["conta"],
                                   "bate a régua": bool(r["conta"] and r["a"] > r["b"])}
verdict = pd.DataFrame(linhas).T
print(verdict)
print("\nContinua a bater a régua:", "SIM" if verdict["bate a régua"].all() else "NÃO")

                  escolhido     régua conta bate a régua
todos      codigo  0.783482  0.263237  True         True
           classe  0.841105  0.155976  True         True
texto novo codigo  0.751404  0.237187  True         True
           classe  0.821808  0.148424  True         True

Continua a bater a régua: SIM


## Conclusão

A melhor das 54 combinações é **exatamente a do E4-S5** (N = 10, 24 épocas, τ = 0,25), e o treino em vigor, repetido, deu sugestões 100% iguais às do E4-S5. Nenhuma outra combinação a bate no top-5 do código completo: o N = 5 com 24 épocas chega perto, mas fica abaixo, e o N = 20 é o pior dos três (ver a tabela e a lista de concorrentes acima). Decisão: **mantém-se o N = 10** (decisão 9 do `qualidade_dados.md`), com as 24 épocas e o τ = 0,25 que já estavam em vigor.

**Limitação (viés de seleção):** com 54 combinações, alguma ganharia só por sorte; o número do vencedor fica otimista e confirma-se no teste, na fase 5. Aqui o efeito é pequeno, porque o vencedor coincide com a opção que já estava em vigor — não foi escolhido à custa de muitas comparações.